# Data Librarian - Modul 2 PyTerrier Tutorial
## This notebook is based on a CIKM workshop
https://github.com/terrier-org/cikm2021tutorial/blob/main/notebooks

In [2]:
#learn more about pyterrier: https://pyterrier.readthedocs.io/en/latest/

In [4]:
#install pyterrier
%pip install python-terrier

Note: you may need to restart the kernel to use updated packages.


In [6]:
%pip install nltk

Note: you may need to restart the kernel to use updated packages.


In [8]:
import pyterrier as pt
import requests
import pandas as pd
import pickle
import nltk
import numpy as np

In [9]:
query = "artificial intelligence"
url =  f"https://www.bibsonomy.org/json/search/{query}?items=1000"

In [10]:
url

'https://www.bibsonomy.org/json/search/artificial intelligence?items=1000'

In [11]:
res = requests.get(url)

In [15]:
data = res.json()

In [16]:
df_ai = pd.DataFrame(data["items"])
df_ai = df_ai[df_ai["type"] == "Publication"]

In [20]:
df_ai

,type,id,tags,intraHash,label,user,description,date,changeDate,count,...,position,category,impact,quartile,dnbtitleid,pdf,broken,organisation,citedreferences,optseries
1000,Publication,https://www.bibsonomy.org/bibtex/24e338b04bc2a...,[710-714],4e338b04bc2abafd9fa0a1f4fe79103a,Artificial Intelligence in Agriculture,ijtsrd,,2021-04-13 13:15:04,2021-04-13 13:15:04,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1001,Publication,https://www.bibsonomy.org/bibtex/283c4174f111d...,"[digitalisation, artificial_intelligence, soci...",83c4174f111d968e8ca51b252f8fea3e,"Artificial intelligence, labour and society",meneteqel,,2024-04-08 09:45:09,2024-04-08 09:46:54,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1002,Publication,https://www.bibsonomy.org/bibtex/29f3b59303571...,"[Artificial, Intelligence, Risks, and, Benefits]",9f3b593035714a894c265f4fb9c535e7,Artificial Intelligence Benefit and Risks,ijtsrd,,2020-05-14 10:40:49,2020-05-14 10:40:49,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1003,Publication,https://www.bibsonomy.org/bibtex/2a97c2267cfe4...,"[industry, artificial_intelligence, health_and...",a97c2267cfe4a39d368f1507149affd0,Artificial intelligence in Industry 4.0: Impli...,meneteqel,,2024-08-01 15:21:53,2024-08-01 15:21:53,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1004,Publication,https://www.bibsonomy.org/bibtex/2df98b0d20723...,"[Human, Intelligence,, intelligence,, centric,...",df98b0d20723566bfc445b3da9e69ac2,Evolution of Artificial Intelligence by Prompt...,ijtsrd,,2024-12-30 07:46:23,2024-12-30 07:46:23,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1995,Publication,https://www.bibsonomy.org/bibtex/20eb599f35759...,[dblp],0eb599f35759d6724fa55f4552f0eeec,Securing Intelligent Autonomous Systems Throug...,dblp,,2023-03-10 00:00:00,2024-04-09 12:57:10,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1996,Publication,https://www.bibsonomy.org/bibtex/259c82e49dd3c...,"[image, artificial, design,, and, human-comput...",59c82e49dd3c187658e7019e2c58f8c8,An Analysis on the Use of Image Design with Ge...,ijtsrd,,2024-03-08 07:51:43,2024-03-08 07:51:43,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1997,Publication,https://www.bibsonomy.org/bibtex/24ccc5d8128ef...,[dblp],4ccc5d8128ef4b4ba28594fdcee31bed,Analysis of Artificial Intelligence Applied in...,dblp,,2023-01-10 00:00:00,2024-04-10 01:50:25,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1998,Publication,https://www.bibsonomy.org/bibtex/268db9fb41250...,[dblp],68db9fb412509b02d6b0cba5eb2876a9,Tenth Scandinavian Conference on Artificial In...,dblp,,2021-02-15 00:00:00,2024-04-09 20:31:41,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
#drops columns where at least 50% of the rows have missing values
df_ai_filtered = df_ai.dropna(axis=1, thresh=len(df_ai)*0.5)

In [24]:
df_ai_filtered.columns

Index(['type', 'id', 'tags', 'intraHash', 'label', 'user', 'description',
       'date', 'changeDate', 'count', 'url', 'interHash', 'pub-type', 'year',
       'author', 'authors', 'volume', 'pages', 'bibtexKey', 'publisher'],
      dtype='object')

In [26]:
#lets only use columns which seems useful for us
useful_coulmns = ['id', 'tags', 'label', 'description', 'date', 'changeDate', 'url', 'pub-type', 'year', 'author', 'authors']
df_ai_filtered = df_ai_filtered[useful_coulmns]

In [28]:
#initialise pyterrier
if pt.java.started():
    pt.java.init()

In [30]:
#pyterrier exspects a docno field
df_ai_filtered['docno'] = df_ai_filtered['id']
df_ai_filtered['text'] = df_ai_filtered['label']

In [32]:
#excursus indexing

In [34]:
#now we start to index our data
#more details at https://pyterrier.readthedocs.io/en/latest/terrier-indexing.html

index_folder = "./ai_index"

indexer = pt.DFIndexer(index_folder, overwrite=True)
index_ref = indexer.index(df_ai_filtered['text'], df_ai_filtered['docno'])

print(f"path to our index: {index_ref.toString()}")

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2259/2741070391.py:6: DeprecationWarning: Call to deprecated class DFIndexer. (use pt.terrier.IterDictIndexer().index(dataframe.to_dict(orient='records')) instead) -- Deprecated since version 0.11.0.
  indexer = pt.DFIndexer(index_folder, overwrite=True)
Java started (triggered by TerrierIndexer.__init__) and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


path to our index: ./ai_index/data.properties


In [36]:
#load our index
index = pt.IndexFactory.of(index_ref)

In [38]:
print(index.getCollectionStatistics().toString())

Number of documents: 1000
Number of terms: 1420
Number of postings: 6189
Number of fields: 0
Number of tokens: 6417
Field names: []
Positions:   false



In [40]:
#term -> term_id Nt Tf
#Nt: In how many documents does the term occur
#TF: How often does the term occur in total

for kv in index.getLexicon():
    print(f"{kv.getKey()} -> {kv.getValue().toString()}")

0 -> term12 Nt=5 TF=6 maxTF=2 @{0 0 0}
02 -> term571 Nt=1 TF=1 maxTF=1 @{0 8 1}
1 -> term1013 Nt=1 TF=1 maxTF=1 @{0 10 1}
10 -> term1042 Nt=2 TF=2 maxTF=1 @{0 12 5}
10th -> term518 Nt=2 TF=2 maxTF=1 @{0 16 5}
11 -> term1101 Nt=1 TF=1 maxTF=1 @{0 20 7}
11th -> term1176 Nt=4 TF=4 maxTF=1 @{0 23 3}
12 -> term477 Nt=1 TF=1 maxTF=1 @{0 31 1}
12th -> term1097 Nt=4 TF=4 maxTF=1 @{0 33 1}
13 -> term1174 Nt=1 TF=1 maxTF=1 @{0 40 3}
13th -> term1249 Nt=3 TF=3 maxTF=1 @{0 42 7}
14 -> term1041 Nt=1 TF=1 maxTF=1 @{0 48 7}
14th -> term1043 Nt=4 TF=4 maxTF=1 @{0 51 3}
15th -> term1040 Nt=4 TF=4 maxTF=1 @{0 58 7}
16 -> term1318 Nt=1 TF=1 maxTF=1 @{0 65 5}
16th -> term901 Nt=4 TF=4 maxTF=1 @{0 68 1}
17 -> term1173 Nt=1 TF=1 maxTF=1 @{0 75 3}
17th -> term958 Nt=4 TF=4 maxTF=1 @{0 77 7}
18 -> term1257 Nt=2 TF=2 maxTF=1 @{0 84 7}
185 -> term712 Nt=2 TF=2 maxTF=1 @{0 88 7}
18th -> term572 Nt=3 TF=3 maxTF=1 @{0 91 5}
19 -> term209 Nt=4 TF=4 maxTF=1 @{0 97 7}
1955 -> term974 Nt=1 TF=1 maxTF=1 @{0 105 3}
1970

In [42]:
term = "game"
index.getLexicon()[term].toString()

'term66 Nt=12 TF=13 maxTF=2 @{0 3346 6}'

In [44]:
#how often do the terms occur
term_freq_dict = {}

for kv in index.getLexicon():
    term_freq_dict[kv.getKey()] = kv.getValue().frequency

In [46]:
#sort all terms from the index in descending order 
term_freq_dict_sorted = sorted(term_freq_dict.items(), key=lambda x: x[1], reverse=True)

In [48]:
#show the 30 nost popular entries
top_k = 30
term_freq_dict_sorted[:top_k]

[('intellig', 963),
 ('artifici', 929),
 ('confer', 72),
 ('proceed', 65),
 ('ai', 64),
 ('system', 63),
 ('base', 55),
 ('learn', 51),
 ('applic', 49),
 ('approach', 45),
 ('us', 44),
 ('gener', 40),
 ('human', 40),
 ('research', 36),
 ('comput', 35),
 ('educ', 33),
 ('develop', 28),
 ('nation', 28),
 ('analysi', 27),
 ('review', 26),
 ('technolog', 26),
 ('advanc', 24),
 ('model', 24),
 ('impact', 23),
 ('intern', 23),
 ('machin', 23),
 ('modern', 23),
 ('explain', 22),
 ('futur', 22),
 ('data', 20)]

In [50]:
#define our search engine
search_engine = pt.BatchRetrieve(index, wmodel="Tf")

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2259/2780833963.py:2: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  search_engine = pt.BatchRetrieve(index, wmodel="Tf")


In [52]:
search_engine.search("system")

,qid,docid,docno,rank,score,query
0,1,292,https://www.bibsonomy.org/bibtex/24768ef52a6c0...,0,2.0,system
1,1,41,https://www.bibsonomy.org/bibtex/249f58604f738...,1,1.0,system
2,1,68,https://www.bibsonomy.org/bibtex/2f693b1d1c623...,2,1.0,system
3,1,81,https://www.bibsonomy.org/bibtex/28585af0407b9...,3,1.0,system
4,1,84,https://www.bibsonomy.org/bibtex/2f6f3aecbaa29...,4,1.0,system
...,...,...,...,...,...,...
57,1,910,https://www.bibsonomy.org/bibtex/2f951427cc281...,57,1.0,system
58,1,957,https://www.bibsonomy.org/bibtex/293fb4fd88de7...,58,1.0,system
59,1,984,https://www.bibsonomy.org/bibtex/20dac3cd23401...,59,1.0,system
60,1,987,https://www.bibsonomy.org/bibtex/234996efa89f2...,60,1.0,system


So the `search()` method returns a dataframe with columns:
 - `qid`: this is by default "1", since it's our first and only query
 - `docid`: Terrier' internal integer for each document
 - `docno`: the external (string) unique identifier for each document
 - `score`: since we use the `Tf` weighting model, this score corresponds the total frequency of the query (terms) in each document
 - `rank`: A handy attribute showing the descending order by score
 - `query`: the input query

As expected, the `Tf` weighting model used here only counts the frequencies of the query terms in each document, i.e.:
$$
score(d,q) = \sum_{t \in q} tf_{t,d}
$$

In [55]:
search_engine.search("intelligent education")

,qid,docid,docno,rank,score,query
0,1,740,https://www.bibsonomy.org/bibtex/2b476e9f37ca1...,0,5.0,intelligent education
1,1,242,https://www.bibsonomy.org/bibtex/2f91c0697115d...,1,3.0,intelligent education
2,1,346,https://www.bibsonomy.org/bibtex/26895e4605d5e...,2,3.0,intelligent education
3,1,348,https://www.bibsonomy.org/bibtex/238a0ee82e84c...,3,3.0,intelligent education
4,1,564,https://www.bibsonomy.org/bibtex/2362d4d486a4a...,4,3.0,intelligent education
...,...,...,...,...,...,...
870,1,989,https://www.bibsonomy.org/bibtex/206f636f05a30...,870,1.0,intelligent education
871,1,991,https://www.bibsonomy.org/bibtex/26db4e5961ea7...,871,1.0,intelligent education
872,1,992,https://www.bibsonomy.org/bibtex/284f798116185...,872,1.0,intelligent education
873,1,998,https://www.bibsonomy.org/bibtex/268db9fb41250...,873,1.0,intelligent education


In [57]:
#what do we need for the inverse document frequency?

In [59]:
from nltk.stem import PorterStemmer
stemmer = PorterStemmer()

In [61]:
#stem the term its base form
stem = stemmer.stem("learning")
stem

'learn'

In [63]:
#in how many documents does the stem 'learn' occur?
lexicon = index.getLexicon()
lexicon[stem].getDocumentFrequency()

49

In [65]:
#how many documents occur in our index?
index.getCollectionStatistics().numberOfDocuments

1000

In [67]:
def get_idf_for_term(term, index):
  lex = index.getLexicon()
  stemmed_term = stemmer.stem(term)

  if not stemmed_term in lex:
    return 
    
  lex_entry = lex[stemmed_term]

  df_term = lex_entry.getDocumentFrequency()
  N = index.getCollectionStatistics().numberOfDocuments
  
  #inverse document frequency
  idf = N/df_term

  #apply logarithm(base 10) to idf
  log_idf = np.emath.logn(10, idf)

  return log_idf

In [69]:
get_idf_for_term("learning", index)

1.3098039199714862

In [71]:
#define your own tf_idf method here:

def calc_tf_idf(query, docno, index):
    #remember that tfidf is the product of two components
    #hint: the tf model search result contains tf frequencies
    return 0

In [73]:
search_engine_tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")

/var/folders/88/9blsh45918g8y3n5vl6vztxr0000gq/T/ipykernel_2259/2880449899.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  search_engine_tfidf = pt.BatchRetrieve(index, wmodel="TF_IDF")


In [75]:
search_engine_tfidf.search("intelligent education")

,qid,docid,docno,rank,score,query
0,1,837,https://www.bibsonomy.org/bibtex/29cc0434b3e50...,0,4.261447,intelligent education
1,1,943,https://www.bibsonomy.org/bibtex/275d5c245ba41...,1,4.261447,intelligent education
2,1,983,https://www.bibsonomy.org/bibtex/2eb8e8e572869...,2,4.261447,intelligent education
3,1,24,https://www.bibsonomy.org/bibtex/259133e526641...,3,3.940288,intelligent education
4,1,78,https://www.bibsonomy.org/bibtex/259133e526641...,4,3.940288,intelligent education
...,...,...,...,...,...,...
870,1,133,https://www.bibsonomy.org/bibtex/2c4df47ce17b7...,870,0.372299,intelligent education
871,1,231,https://www.bibsonomy.org/bibtex/29fcc0f646278...,871,0.372299,intelligent education
872,1,600,https://www.bibsonomy.org/bibtex/21b69569af8d0...,872,0.372299,intelligent education
873,1,251,https://www.bibsonomy.org/bibtex/2ba15da18da2c...,873,0.344993,intelligent education


In [77]:
#Now we want to build an index with multiple fields

In [79]:
df_ai_filtered.columns

Index(['id', 'tags', 'label', 'description', 'date', 'changeDate', 'url',
       'pub-type', 'year', 'author', 'authors', 'docno', 'text'],
      dtype='object')

In [81]:
#transfrom dataframe into list of dictionaries
ai_dict = df_ai_filtered.to_dict(orient='records')

In [83]:
ai_dict

[{'id': 'https://www.bibsonomy.org/bibtex/24e338b04bc2abafd9fa0a1f4fe79103a/ijtsrd',
  'tags': ['710-714'],
  'label': 'Artificial Intelligence in Agriculture',
  'description': '',
  'date': '2021-04-13 13:15:04',
  'changeDate': '2021-04-13 13:15:04',
  'url': 'https://www.ijtsrd.com/engineering/electrical-engineering/38513/artificial-intelligence-in-agriculture/matthew-n-o-sadiku',
  'pub-type': 'article',
  'year': '2021',
  'author': ['Matthew N. O. Sadiku | Sarhan M. Musa | Abayomi Ajayi-Majebi'],
  'authors': [{'first': 'Matthew N. O. Sadiku | Sarhan M. Musa | Abayomi',
    'last': 'Ajayi-Majebi'}],
  'docno': 'https://www.bibsonomy.org/bibtex/24e338b04bc2abafd9fa0a1f4fe79103a/ijtsrd',
  'text': 'Artificial Intelligence in Agriculture'},
 {'id': 'https://www.bibsonomy.org/bibtex/283c4174f111d968e8ca51b252f8fea3e/meneteqel',
  'tags': ['digitalisation',
   'artificial_intelligence',
   'society',
   'labour_rights'],
  'label': 'Artificial intelligence, labour and society',
  'de

In [93]:
index_folder_mult = "./ai_index_mult"

fields=['docno', 'text', 'tags', 'description']

indexer_mult = pt.IterDictIndexer(index_folder_mult, meta={'docno': 200, 'text': 4096}, overwrite=True)
index_ref_mult = indexer_mult.index(ai_dict)

print(f"path to our index: {index_ref_mult.toString()}")

path to our index: ./ai_index_mult/data.properties


In [95]:
index_mult = pt.IndexFactory.of(index_ref_mult)

In [97]:
print(index_mult.getCollectionStatistics().toString())

Number of documents: 1000
Number of terms: 1420
Number of postings: 6189
Number of fields: 1
Number of tokens: 6417
Field names: [text]
Positions:   false



In [99]:
search_engine_mult = pt.terrier.Retriever(index_mult, wmodel="TF_IDF")

In [101]:
res = search_engine_mult.search("intelligence")
res

,qid,docid,docno,rank,score,query
0,1,508,https://www.bibsonomy.org/bibtex/2f7cf941158e2...,0,0.969908,intelligence
1,1,739,https://www.bibsonomy.org/bibtex/21a922cb4af1f...,1,0.969908,intelligence
2,1,738,https://www.bibsonomy.org/bibtex/224ac67b53792...,2,0.955768,intelligence
3,1,604,https://www.bibsonomy.org/bibtex/247f921d7eaa8...,3,0.922362,intelligence
4,1,610,https://www.bibsonomy.org/bibtex/23097df37203d...,4,0.922362,intelligence
...,...,...,...,...,...,...
870,1,133,https://www.bibsonomy.org/bibtex/2c4df47ce17b7...,870,0.372299,intelligence
871,1,231,https://www.bibsonomy.org/bibtex/29fcc0f646278...,871,0.372299,intelligence
872,1,600,https://www.bibsonomy.org/bibtex/21b69569af8d0...,872,0.372299,intelligence
873,1,251,https://www.bibsonomy.org/bibtex/2ba15da18da2c...,873,0.344993,intelligence


In [103]:
#save our data
pickle.dump(df_ai_filtered, open("ai_index_mult/ai_publications.pkl", "wb"))

In [105]:
#install steamlit
%pip install streamlit

Note: you may need to restart the kernel to use updated packages.
